In [2]:
import pandas as pd
orders = pd.read_csv('olist_orders_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')

In [3]:
print(orders.shape)
print(reviews.shape)
print(items.shape)
print(products.shape)

(99441, 8)
(99224, 7)
(112650, 7)
(32951, 9)


In [4]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [5]:
orders['order_status'].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [6]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]

for cols in date_cols:
  orders[cols] = pd.to_datetime(orders[cols])

In [7]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [8]:
delivered = orders[orders['order_status'] == 'delivered'].copy()
delivered['delay_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_estimated_delivery_date']
    ).dt.days


In [9]:
# delivered['delay_days'].describe()
# print(delivered)
# Easy method
#pct_late_delivered_orders = (delivered['delay_days'] > 0).mean() * 100
late_delivered_orders = delivered[delivered['delay_days'] > 0]
pct_late_delivered_orders = ((late_delivered_orders['order_id'].count() * 100 )/ delivered['order_id'].count())
print(pct_late_delivered_orders)


6.77252845208234


In [10]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [11]:
delivered['is_late'] = delivered['delay_days'] > 0
delivered_rating = reviews.merge(delivered, on = 'order_id', how = 'inner')
# print(len(delivered_rating), len(delivered))
Avg_rating = (
    delivered_rating
    .groupby('is_late')['review_score'].mean()
)

In [12]:
delivered_rating['delay_bucket'] = pd.cut(
    delivered_rating['delay_days'],
    bins = [-999,-10,0,5,15,999],
    labels = ['>10d early', '0-10d early', '1-5d late', '6-15d late', '>15d late']
)

In [13]:
delivered_rating.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96361 entries, 0 to 96360
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   review_id                      96361 non-null  object        
 1   order_id                       96361 non-null  object        
 2   review_score                   96361 non-null  int64         
 3   review_comment_title           11208 non-null  object        
 4   review_comment_message         39099 non-null  object        
 5   review_creation_date           96361 non-null  object        
 6   review_answer_timestamp        96361 non-null  object        
 7   customer_id                    96361 non-null  object        
 8   order_status                   96361 non-null  object        
 9   order_purchase_timestamp       96361 non-null  datetime64[ns]
 10  order_approved_at              96347 non-null  datetime64[ns]
 11  order_delivered

In [14]:
avg_delay_rating = delivered_rating.groupby('delay_bucket', observed=True)['review_score'].mean()
print(avg_delay_rating)

delay_bucket
>10d early     4.322943
0-10d early    4.217284
1-5d late      2.987541
6-15d late     1.745586
>15d late      1.727273
Name: review_score, dtype: float64


In [15]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [16]:
items_with_category = items.merge(products, on='product_id', how='left')

In [17]:
items_rated_with_category = items_with_category.merge(delivered_rating, on = 'order_id', how = 'inner')

In [18]:
items_rated_with_category.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110013 entries, 0 to 110012
Data columns (total 31 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110013 non-null  object        
 1   order_item_id                  110013 non-null  int64         
 2   product_id                     110013 non-null  object        
 3   seller_id                      110013 non-null  object        
 4   shipping_limit_date            110013 non-null  object        
 5   price                          110013 non-null  float64       
 6   freight_value                  110013 non-null  float64       
 7   product_category_name          108480 non-null  object        
 8   product_name_lenght            108480 non-null  float64       
 9   product_description_lenght     108480 non-null  float64       
 10  product_photos_qty             108480 non-null  float64       
 11  

In [20]:
items_rated_per_category = (
    items_rated_with_category
    .groupby('product_category_name')
    .agg(
        avg_review_score=('review_score', 'mean'),
        avg_delay=('delay_days', 'mean'),
        total_cnt=('order_id', 'count')
    )
    .reset_index()
    .query('total_cnt >= 500')
    .sort_values('avg_review_score')
)

print(items_rated_per_category.head(10))

     product_category_name  avg_review_score  avg_delay  total_cnt
55       moveis_escritorio          3.517428 -11.825721       1664
13         cama_mesa_banho          3.920983 -11.689941      10985
54        moveis_decoracao          3.950116 -12.485354       8159
16         casa_construcao          3.959528 -11.470489        593
44  informatica_acessorios          3.984750 -12.468779       7672
70               telefonia          3.995009 -11.391107       4408
30             eletronicos          4.067503 -11.154187       2711
66      relogios_presentes          4.071931 -11.968401       5825
9                    bebes          4.079205 -11.733401       2967
40      ferramentas_jardim          4.081335 -12.031265       4254


In [22]:
one_star = delivered_rating[delivered_rating['review_score'] == 1]
one_star_late = one_star[one_star['is_late'] == True]
pct_one_star_islate = (len(one_star_late) * 100)/ len(one_star)
print(pct_one_star_islate)

36.61492664256858
